# Step 1: data and linear probes

One-time setup:
1. Make a folder `grsats` in My Drive and upload `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip`, and `finalfitz17k.zip` (the `finalfitz17k` folder zipped) into it.
2. Make a GitHub token with read access to the repo and add it in Colab under Secrets (key icon) as `GH_TOKEN`.
3. Runtime > Change runtime type > T4 GPU.

In [1]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

Mounted at /content/drive
/content/repo/code


In [2]:
# unzip images to the colab disk, reading thousands of files straight from drive is really slow
# (same folders you get on a mac by double clicking the zips)
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python -m scripts.check_data
!python -m pytest -q tests


ham: 9,948 images
  train        8,164 (1,740, 21.3%)
  val          568 (63, 11.1%)
  test         1,216 (149, 12.3%)
  pool         1,784 (212, 11.9%)
  pool female  820 (80, 9.8%)
  pool male    964 (132, 13.7%)
  pool 60+     578 (150, 26.0%)
  pool <60     1,206 (62, 5.1%)
  missing image files: 0

fitz: 11,098 images
  train        7,779 (1,144, 14.7%)
  val          1,100 (154, 14.0%)
  test         2,219 (325, 14.6%)
  pool         3,319 (479, 14.4%)
  pool I-II    1,794 (287, 16.0%)
  pool III-IV  1,163 (160, 13.8%)
  pool V-VI    362 (32, 8.8%)
  missing image files: 0
.......                                                                  [100%]
7 passed in 2.37s


In [3]:
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.extract_features --dataset {ds} --backbone {bb}

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth
100% 330M/330M [00:01<00:00, 326MB/s]
100% 156/156 [01:47<00:00,  1.46it/s]
saved (9948, 768) to /content/drive/MyDrive/grsats/outputs/ham_dinov2_features.npz
open_clip_config.jso

In [4]:
# cpu only, could also run this locally after downloading the outputs folder
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.train_probes --dataset {ds} --backbone {bb}


seed 0: l2 = 100, dev nll = 0.317
 group    n   auc   nll
   all 1784 0.959 0.167
female  820 0.968 0.137
  male  964 0.950 0.192
   60+  578 0.915 0.317
   <60 1206 0.966 0.095

seed 1: l2 = 100, dev nll = 0.325
 group    n   auc   nll
   all 1784 0.959 0.166
female  820 0.964 0.141
  male  964 0.954 0.188
   60+  578 0.916 0.314
   <60 1206 0.968 0.095

seed 2: l2 = 100, dev nll = 0.339
 group    n   auc   nll
   all 1784 0.959 0.166
female  820 0.966 0.138
  male  964 0.952 0.190
   60+  578 0.915 0.316
   <60 1206 0.966 0.094

ham_dinov2 mean over seeds:
          auc           nll       
         mean    std   mean    std
group                             
all     0.959  0.000  0.166  0.000
female  0.966  0.002  0.139  0.002
male    0.952  0.002  0.190  0.002
60+     0.915  0.001  0.316  0.001
<60     0.967  0.001  0.095  0.001

seed 0: l2 = 0.01, dev nll = 0.309
 group    n   auc   nll
   all 1784 0.949 0.175
female  820 0.947 0.161
  male  964 0.950 0.187
   60+  578 0.915 0.32